# A_s189 -> operative notes for HeiCo-FOCUS (Colab)

Runs **A_s189** (Qwen3.5-9B + LoRA, merged) over the HeiCo-FOCUS videos and writes one
**operative note** per video.

Steps in this notebook:

1. Mount Google Drive
2. Copy videos from Drive to session storage (`/content/videos`)
3. Download the base model + A_s189 weights and assemble the **full A_s189**
4. Run inference and write the notes

A_s189 is a single-frame VQA model with no cross-frame memory, so each video is handled
in two stages with the same model: (1) describe sampled frames, (2) feed the timeline
back as text to write the note.

**Before you run:** read the config cell below. You must tell the notebook where the
A_s189 adapter weights live (Drive / S3 / HF). A >= 24 GB GPU is assumed.


In [ ]:
# 1. Dependencies. Colab ships torch + transformers; ffmpeg is already present.
#    Two belt-and-suspenders fixes for the peft/torchao conflict:
#      (a) pin peft to 0.20.0 (the version A_s189 was saved with);
#      (b) uninstall torchao, so even a newer peft won't trip its torchao>=0.16 check.
%pip install -q "peft==0.20.0" "huggingface_hub>=0.27" "accelerate>=0.34"
%pip uninstall -y torchao
!which ffmpeg >/dev/null 2>&1 || apt-get install -y -qq ffmpeg
!ffmpeg -version | head -1


In [ ]:
# 2. Configuration -- EDIT THIS CELL
import os

# Hugging Face token: base model may be gated; also used if the adapter is on HF.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("Hugging Face token: ")
os.environ["HF_TOKEN"] = HF_TOKEN

BASE_MODEL_ID = "Qwen/Qwen3.5-9B"

# Where the A_s189 adapter weights live. One of: "drive" | "s3" | "hf".
ADAPTER_SOURCE = "drive"
ADAPTER_DRIVE_PATH = "/content/drive/MyDrive/Debjoti/orena-procedure/A_s189"  # adapter_config.json + adapter_model.safetensors
ADAPTER_HF_REPO = ""          # e.g. "yourname/A_s189"  (used when ADAPTER_SOURCE == "hf")
S3_ADAPTER_URI = "s3://stanford-train-data/models/A_s189"

# Videos
DRIVE_VIDEO_DIR = "/content/drive/MyDrive/Debjoti/orena-procedure/heico-focus-vqa/videos"
SESSION_VIDEO_DIR = "/content/videos"
MAX_VIDEOS = 0      # 0 = copy all; set 1 for a pilot
ONLY = ""           # copy only videos whose filename contains this substring

OUTPUT_DIR = "/content/notes"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Debjoti/orena-procedure/notes"  # where notes are copied back

SAMPLE_INTERVAL_SEC = 5    # one frame every N seconds
BATCH_SIZE = 8


In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 4. Copy videos from Drive to session storage (fast local SSD for ffmpeg/frame reads)
import glob, os, shutil

VIDEO_EXTS = (".avi", ".mp4", ".mov", ".mkv", ".webm", ".mpg", ".mpeg")

print("Drive mounted:", os.path.isdir("/content/drive"))
print("Video dir exists:", os.path.isdir(DRIVE_VIDEO_DIR), "->", DRIVE_VIDEO_DIR)

os.makedirs(SESSION_VIDEO_DIR, exist_ok=True)
src = sorted(p for p in glob.glob(os.path.join(DRIVE_VIDEO_DIR, "**", "*"), recursive=True)
             if p.lower().endswith(VIDEO_EXTS))

if not src:
    print("No video-like files found. Listing what IS there:")
    for base in (DRIVE_VIDEO_DIR, os.path.dirname(DRIVE_VIDEO_DIR),
                 "/content/drive/MyDrive/Debjoti/orena-procedure"):
        if os.path.isdir(base):
            print(f"  {base}/")
            for name in sorted(os.listdir(base))[:50]:
                kind = "dir " if os.path.isdir(os.path.join(base, name)) else "file"
                print(f"    [{kind}] {name}")

if ONLY:
    src = [p for p in src if ONLY in os.path.basename(p)]
if MAX_VIDEOS:
    src = src[:MAX_VIDEOS]

total_gb = sum(os.path.getsize(p) for p in src) / 1e9
print(f"{len(src)} video(s), {total_gb:.1f} GB total -> {SESSION_VIDEO_DIR}")
print(f"Current /content free space: {shutil.disk_usage('/content').free / 1e9:.1f} GB")

for p in src:
    dst = os.path.join(SESSION_VIDEO_DIR, os.path.basename(p))
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(p):
        continue
    shutil.copy2(p, dst)
    print(f"  copied {os.path.basename(p)}")
print("copy done.")


In [ ]:
# 5. Download base model + A_s189 weights, merge into the full A_s189
import os, shutil
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

ADAPTER_DIR = "/content/models/A_s189"

print("Downloading base model:", BASE_MODEL_ID)
processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    padding_side="left",
    size={"longest_edge": 1048576, "shortest_edge": 65536},
)
base = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="sdpa",
)

print("Downloading A_s189 adapter from:", ADAPTER_SOURCE)
os.makedirs(ADAPTER_DIR, exist_ok=True)
if ADAPTER_SOURCE == "drive":
    for fn in os.listdir(ADAPTER_DRIVE_PATH):
        shutil.copy2(os.path.join(ADAPTER_DRIVE_PATH, fn), ADAPTER_DIR)
elif ADAPTER_SOURCE == "s3":
    # Requires AWS credentials configured in this session, e.g. from Colab secrets:
    #   os.environ["AWS_ACCESS_KEY_ID"] = ...
    #   os.environ["AWS_SECRET_ACCESS_KEY"] = ...
    r = os.system(f"aws s3 cp {S3_ADAPTER_URI}/ {ADAPTER_DIR}/ --recursive")
    if r != 0:
        raise RuntimeError("aws s3 cp failed; set AWS credentials first")
elif ADAPTER_SOURCE == "hf":
    from huggingface_hub import snapshot_download
    snapshot_download(ADAPTER_HF_REPO, token=HF_TOKEN, local_dir=ADAPTER_DIR)
else:
    raise ValueError("ADAPTER_SOURCE must be drive/s3/hf")

print("Files in adapter dir:", sorted(os.listdir(ADAPTER_DIR)))

print("Merging LoRA into base -> full A_s189 ...")
from peft import PeftModel
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model = model.merge_and_unload()
model.eval()
model.config.use_cache = True

# EOS ids: Qwen3.5 ships no generation_config.json. See generate_notes.py.
tok = processor.tokenizer
EOS_IDS = sorted({tok.convert_tokens_to_ids("<|im_end|>"),
                  tok.convert_tokens_to_ids("<|endoftext|>")})
print("A_s189 ready. EOS:", EOS_IDS)


In [ ]:
# 6. The two-stage pipeline (ported from operative-notes/generate_notes.py)
import subprocess, tempfile, shutil
from pathlib import Path
from PIL import Image
import torch

SYSTEM_FRAME = "You are a surgical assistant observing a laparoscopic procedure."
FRAME_QUESTION = "Describe what is happening in this surgical frame in one or two sentences."
SYSTEM_NOTE = "You are a surgical resident writing an operative note."
NOTE_QUESTION_TMPL = (
    "Write a complete operative note for this procedure, based on the timeline of "
    "frame descriptions below. Include the sections Indication, Procedure, Findings "
    "and Closure, written in full prose sentences like a real operative report.\n\n"
    "Timeline:\n{timeline}"
)
FRAME_DESC_MAX_NEW_TOKENS = 128
NOTE_MAX_NEW_TOKENS = 2048
MAX_TIMELINE_CHARS = 12000


def _fmt_ts(seconds):
    s = int(round(seconds))
    return f"{s // 3600:02d}:{(s % 3600) // 60:02d}:{s % 60:02d}"


def _build_prompt(question, system, with_image):
    q = str(question).replace("<image>", "").strip()
    user = ([{"type": "image"}, {"type": "text", "text": q}]
            if with_image else [{"type": "text", "text": q}])
    msgs = [{"role": "system", "content": [{"type": "text", "text": system}]},
            {"role": "user", "content": user}]
    return processor.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)


def _generate(prompts, images, max_new_tokens):
    if images is not None:
        batch = processor(text=prompts, images=images, return_tensors="pt",
                          padding=True).to(model.device)
    else:
        batch = processor(text=prompts, return_tensors="pt", padding=True).to(model.device)
    n_in = batch["input_ids"].shape[1]
    with torch.inference_mode():
        gen = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False,
                             num_beams=1, eos_token_id=EOS_IDS,
                             pad_token_id=processor.tokenizer.pad_token_id)
    return processor.tokenizer.batch_decode(gen[:, n_in:], skip_special_tokens=True)


def extract_frames(video_path, interval_sec, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    fps = 1.0 / interval_sec
    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-i", str(video_path),
           "-vf", f"fps={fps:.6f}", "-q:v", "3", str(out_dir / "frame_%06d.jpg")]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip()[:500])
    frames = sorted(out_dir.glob("frame_*.jpg"))
    if not frames:
        raise RuntimeError("no frames")
    return frames


def _describe_frames(frames, batch_size):
    prompt = _build_prompt(FRAME_QUESTION, SYSTEM_FRAME, True)
    out = []
    for s in range(0, len(frames), batch_size):
        chunk = frames[s:s + batch_size]
        images, ok = [], []
        for fp in chunk:
            try:
                images.append(Image.open(fp).convert("RGB"))
                ok.append(fp)
            except Exception:
                pass
        if not images:
            continue
        texts = _generate([prompt] * len(images), images, FRAME_DESC_MAX_NEW_TOKENS)
        out.extend(zip(ok, texts))
    return out


def _compress_timeline(lines):
    if sum(len(l) + 1 for l in lines) <= MAX_TIMELINE_CHARS:
        return lines
    lo, hi, stride = 1, len(lines), 1
    while lo <= hi:
        mid = (lo + hi) // 2
        chosen = lines[::mid]
        if chosen[-1] != lines[-1]:
            chosen = chosen + [lines[-1]]
        if sum(len(l) + 1 for l in chosen) <= MAX_TIMELINE_CHARS:
            stride, lo = mid, mid + 1
        else:
            hi = mid - 1
    out = lines[::stride]
    if out[-1] != lines[-1]:
        out.append(lines[-1])
    return out


def _synthesize_note(timeline, anchor):
    q = NOTE_QUESTION_TMPL.format(timeline=timeline)
    try:
        return _generate([_build_prompt(q, SYSTEM_NOTE, False)], None, NOTE_MAX_NEW_TOKENS)[0]
    except Exception:
        return _generate([_build_prompt(q, SYSTEM_NOTE, True)], [anchor], NOTE_MAX_NEW_TOKENS)[0]


def process_video(video_path, interval_sec, batch_size):
    name = Path(video_path).name
    print(f"== {name} ==")
    tmp = Path(tempfile.mkdtemp(prefix="frames_"))
    try:
        frames = extract_frames(video_path, interval_sec, tmp)
        print(f"  extracted {len(frames)} frames")
    except Exception as e:
        shutil.rmtree(tmp, ignore_errors=True)
        return {"video": name, "error": str(e)}
    anchor = None
    try:
        anchor = Image.open(frames[-1]).convert("RGB")
        descs = _describe_frames(frames, batch_size)
    except Exception as e:
        shutil.rmtree(tmp, ignore_errors=True)
        return {"video": name, "error": str(e)}
    lines = [f"[{_fmt_ts(i * interval_sec)}] {t.strip()}"
             for i, (_, t) in enumerate(descs) if t.strip()]
    timeline = "\n".join(_compress_timeline(lines))
    shutil.rmtree(tmp, ignore_errors=True)
    if not timeline.strip():
        return {"video": name, "timeline": "", "note": "(no frame descriptions produced)"}
    print(f"  timeline: {len(timeline)} chars")
    try:
        note = _synthesize_note(timeline, anchor)
    except Exception as e:
        return {"video": name, "timeline": timeline, "error": str(e)}
    return {"video": name, "timeline": timeline, "note": note.strip()}


def write_note(out_dir, result):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    stem = Path(result["video"]).stem
    p = Path(out_dir) / f"{stem}.md"
    with open(p, "w") as f:
        f.write(f"# Operative note — {result['video']}\n\n")
        if result.get("error"):
            f.write(f"**ERROR:** {result['error']}\n\n")
        if result.get("timeline"):
            f.write("## Frame timeline\n\n```\n" + result["timeline"] + "\n```\n\n")
        if result.get("note"):
            f.write("## Operative note\n\n" + result["note"] + "\n")
    print(f"  wrote {p}")


In [ ]:
# 7. Run inference over all copied videos
import glob, os

videos = sorted(glob.glob(os.path.join(SESSION_VIDEO_DIR, "*")))
print(f"{len(videos)} video(s) in {SESSION_VIDEO_DIR}")

for v in videos:
    r = process_video(v, SAMPLE_INTERVAL_SEC, BATCH_SIZE)
    write_note(OUTPUT_DIR, r)
print("All videos done.")


## Alternative: direct video inference (cross-frame context)

The cell below feeds a **window of ordered frames in one prompt** instead of one frame
at a time. The model then sees several consecutive frames together and can reason across
them (order, motion, continuity). This is the practical "video" path for A_s189, whose
Qwen3.5-9B base accepts multiple images per turn.

Context is managed by sampling at low fps (`VIDEO_FPS`) and splitting into windows of
`SEG_FRAMES` frames, then aggregating the window descriptions into the note. It reuses
the helpers from the pipeline cell above, so run that cell first. Pilot it on one video
(`MAX_VIDEOS = 1` in the config cell) and compare `/content/notes` vs `/content/notes_video`.


In [ ]:
# ── Direct video inference (cross-frame context) ──────────────────────────
# Feeds a WINDOW of ordered frames in ONE prompt, so the model sees several
# consecutive frames at once and reasons across them (order, motion, continuity).
#
# Context management: a full operation cannot fit one call, so we
#   1. sample the video at low fps (VIDEO_FPS), then
#   2. split into windows of SEG_FRAMES frames, describe each window,
#   3. aggregate the window descriptions into the note (reuses cell 6 helpers).
import glob, os, subprocess, tempfile, shutil
from pathlib import Path
from PIL import Image
import torch

VIDEO_FPS = 0.5          # frames sampled per second
SEG_FRAMES = 8           # frames per model call (lower it on a context/OOM error)
VIDEO_MAX_NEW_TOKENS = 256
VIDEO_OUTPUT_DIR = OUTPUT_DIR + "_video"

SYSTEM_VIDEO = "You are a surgical assistant watching a laparoscopic procedure."
VIDEO_QUESTION = ("These are {n} consecutive frames from one surgical video. "
                  "Describe the key steps, instruments and events across these frames "
                  "in a few sentences.")

def sample_video_frames(video_path, fps, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-i", str(video_path),
           "-vf", f"fps={fps:.6f}", "-q:v", "3", str(out_dir / "f_%06d.jpg")]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip()[:400])
    return sorted(out_dir.glob("f_*.jpg"))

def build_video_prompt(n):
    user = [{"type": "image"} for _ in range(n)]
    user.append({"type": "text", "text": VIDEO_QUESTION.format(n=n)})
    msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_VIDEO}]},
            {"role": "user", "content": user}]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)

def describe_window(frames):
    images = [Image.open(p).convert("RGB") for p in frames]
    prompt = build_video_prompt(len(images))
    batch = processor(text=[prompt], images=images, return_tensors="pt", padding=True).to(model.device)
    n_in = batch["input_ids"].shape[1]
    with torch.inference_mode():
        gen = model.generate(**batch, max_new_tokens=VIDEO_MAX_NEW_TOKENS, do_sample=False,
                             num_beams=1, eos_token_id=EOS_IDS,
                             pad_token_id=processor.tokenizer.pad_token_id)
    return processor.tokenizer.batch_decode(gen[:, n_in:], skip_special_tokens=True)[0].strip()

def process_video_direct(video_path, fps=VIDEO_FPS, seg=SEG_FRAMES):
    name = Path(video_path).name
    print(f"== (video) {name} ==")
    tmp = Path(tempfile.mkdtemp(prefix="vf_"))
    try:
        frames = sample_video_frames(video_path, fps, tmp)
        print(f"  sampled {len(frames)} frames @ {fps} fps")
    except Exception as e:
        shutil.rmtree(tmp, ignore_errors=True)
        return {"video": name, "error": str(e)}
    anchor = Image.open(frames[-1]).convert("RGB") if frames else None
    seg_secs = 1.0 / fps
    descs = []
    nwin = (len(frames) + seg - 1) // seg
    for wi in range(0, len(frames), seg):
        win = frames[wi:wi + seg]
        t0 = wi * seg_secs
        t1 = min(wi + seg, len(frames)) * seg_secs
        try:
            d = describe_window(win)
        except Exception as e:
            print(f"  window {wi // seg + 1} failed: {e}")
            d = ""
        descs.append((t0, t1, d))
        print(f"  window {wi // seg + 1}/{nwin} [{_fmt_ts(t0)}-{_fmt_ts(t1)}]: {d[:70]}")
    shutil.rmtree(tmp, ignore_errors=True)
    lines = [f"[{_fmt_ts(t0)}-{_fmt_ts(t1)}] {d}" for t0, t1, d in descs if d]
    timeline = "\n".join(_compress_timeline(lines)) if lines else ""
    if not timeline.strip():
        return {"video": name, "timeline": "", "note": "(no segment descriptions produced)"}
    try:
        note = _synthesize_note(timeline, anchor).strip()
    except Exception as e:
        return {"video": name, "timeline": timeline, "error": str(e)}
    return {"video": name, "timeline": timeline, "note": note}

for v in sorted(glob.glob(os.path.join(SESSION_VIDEO_DIR, "*"))):
    write_note(VIDEO_OUTPUT_DIR, process_video_direct(v))
print("done. video notes in", VIDEO_OUTPUT_DIR)


In [ ]:
# 8. (Optional) copy the generated notes back to Google Drive
import shutil, os

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".md"):
        shutil.copy2(os.path.join(OUTPUT_DIR, f), DRIVE_OUTPUT_DIR)
print(f"copied notes to {DRIVE_OUTPUT_DIR}")
